# 1) Import Libraries

In [293]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 2). Load dataset

In [294]:
df=pd.read_csv("/workspaces/Modelling_Credit_Defaults-Logistic_Regression-/data/lending_club_loan_data/loan.csv")
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Columns: 111 entries, id to total_il_high_credit_limit
dtypes: float64(74), int64(13), str(24)
memory usage: 33.6 MB


/tmp/ipykernel_60391/788162185.py:1: DtypeWarning: Columns (0: next_pymnt_d) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("/workspaces/Modelling_Credit_Defaults-Logistic_Regression-/data/lending_club_loan_data/loan.csv")


# 3). Data Cleaning

In [295]:
df.info()
df.duplicated().sum() # Result: No duplicates
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Columns: 111 entries, id to total_il_high_credit_limit
dtypes: float64(74), int64(13), str(24)
memory usage: 33.6 MB


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,1077501,1296599,5000,5000,4975.0,36 months,10.65%,162.87,B,B2,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
1,1077430,1314167,2500,2500,2500.0,60 months,15.27%,59.83,C,C4,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
2,1077175,1313524,2400,2400,2400.0,36 months,15.96%,84.33,C,C5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
3,1076863,1277178,10000,10000,10000.0,36 months,13.49%,339.31,C,C1,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN
4,1075358,1311748,3000,3000,3000.0,60 months,12.69%,67.79,B,B5,...,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN


### Data Dictionary

In [296]:
lc_dkt=pd.read_excel("/workspaces/Modelling_Credit_Defaults-Logistic_Regression-/data/lending_club_loan_data/LCDataDictionary.xlsx")
lc_dkt

,LoanStatNew,Description
0,acc_now_delinq,The number of accounts on which the borrower i...
1,acc_open_past_24mths,Number of trades opened in past 24 months.
2,addr_state,The state provided by the borrower in the loan...
3,all_util,Balance to credit limit on all trades
4,annual_inc,The self-reported annual income provided by th...
...,...,...
148,settlement_amount,The loan amount that the borrower has agreed t...
149,settlement_percentage,The settlement amount as a percentage of the p...
150,settlement_term,The number of months that the borrower will be...
151,NaN,NaN


In [297]:
df.isna().sum()/len(df)*100

id                              0.000000
member_id                       0.000000
loan_amnt                       0.000000
funded_amnt                     0.000000
funded_amnt_inv                 0.000000
                                 ...    
tax_liens                       0.098195
tot_hi_cred_lim               100.000000
total_bal_ex_mort             100.000000
total_bc_limit                100.000000
total_il_high_credit_limit    100.000000
Length: 111, dtype: float64

In [298]:
def wrangle(df):
    # Pick columns that are not 100% null
    cols=["id", "member_id","loan_amnt","funded_amnt","funded_amnt_inv","term",
        "int_rate","installment","grade","sub_grade",
        "emp_title","emp_length","home_ownership","annual_inc",
        "verification_status","issue_d","loan_status","pymnt_plan",
        "url","desc","purpose","title",
        "zip_code","addr_state","dti","delinq_2yrs",
        "earliest_cr_line","inq_last_6mths","mths_since_last_delinq",
        "mths_since_last_record","open_acc","pub_rec","revol_bal",
        "revol_util","total_acc","initial_list_status","out_prncp",
        "out_prncp_inv","total_pymnt","total_pymnt_inv","total_rec_prncp",
        "total_rec_int","total_rec_late_fee","recoveries",
        "collection_recovery_fee","last_pymnt_d","last_pymnt_amnt",
        "next_pymnt_d","last_credit_pull_d","collections_12_mths_ex_med",
        "policy_code","application_type","acc_now_delinq",
        "chargeoff_within_12_mths","delinq_amnt","pub_rec_bankruptcies",
        "tax_liens"
        ]
    df=df[cols]
    
    drop_cols=[]
    
    # Drop high cardinality features
    high_low_cad=["id","member_id","url","zip_code","pymnt_plan",
                  "chargeoff_within_12_mths","delinq_amnt","tax_liens",
                  "policy_code","initial_list_status","application_type",
                  "collections_12_mths_ex_med","acc_now_delinq"]
    drop_cols.extend(high_low_cad)

    # Drop columns with more than 50% Nulls
    null_cols=["mths_since_last_delinq", "mths_since_last_record", "next_pymnt_d"]
    drop_cols.extend(null_cols)

    # Drop Leaky features
    leaky_cols=["total_pymnt","total_pymnt_inv", "last_pymnt_amnt",
                "recoveries", "last_pymnt_d", "collection_recovery_fee",
                "total_rec_late_fee", "total_rec_int", "total_rec_prncp" ]


    drop_cols.extend(leaky_cols)
    


    df.drop(columns=drop_cols, inplace=True)
    return df

In [299]:
df=wrangle(df)
df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,out_prncp,out_prncp_inv,last_credit_pull_d,pub_rec_bankruptcies
0,5000,5000,4975.0,36 months,10.65%,162.87,B,B2,NaN,10+ years,...,1,3,0,13648,83.70%,9,0.00,0.00,May-16,0.0
1,2500,2500,2500.0,60 months,15.27%,59.83,C,C4,Ryder,< 1 year,...,5,3,0,1687,9.40%,4,0.00,0.00,Sep-13,0.0
2,2400,2400,2400.0,36 months,15.96%,84.33,C,C5,NaN,10+ years,...,2,2,0,2956,98.50%,10,0.00,0.00,May-16,0.0
3,10000,10000,10000.0,36 months,13.49%,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,1,10,0,5598,21%,37,0.00,0.00,Apr-16,0.0
4,3000,3000,3000.0,60 months,12.69%,67.79,B,B5,University Medical Group,1 year,...,0,15,0,27783,53.90%,38,524.06,524.06,May-16,0.0


In [300]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39717 entries, 0 to 39716
Data columns (total 32 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   loan_amnt             39717 non-null  int64  
 1   funded_amnt           39717 non-null  int64  
 2   funded_amnt_inv       39717 non-null  float64
 3   term                  39717 non-null  str    
 4   int_rate              39717 non-null  str    
 5   installment           39717 non-null  float64
 6   grade                 39717 non-null  str    
 7   sub_grade             39717 non-null  str    
 8   emp_title             37258 non-null  str    
 9   emp_length            38642 non-null  str    
 10  home_ownership        39717 non-null  str    
 11  annual_inc            39717 non-null  float64
 12  verification_status   39717 non-null  str    
 13  issue_d               39717 non-null  str    
 14  loan_status           39717 non-null  str    
 15  desc                  26775 no

In [301]:
df.nunique().sort_values(ascending=False)

emp_title               28820
desc                    26526
revol_bal               21711
title                   19615
installment             15383
funded_amnt_inv          8205
annual_inc               5318
dti                      2868
out_prncp_inv            1138
out_prncp                1137
revol_util               1089
funded_amnt              1041
loan_amnt                 885
earliest_cr_line          526
int_rate                  371
last_credit_pull_d        106
total_acc                  82
issue_d                    55
addr_state                 50
open_acc                   40
sub_grade                  35
purpose                    14
delinq_2yrs                11
emp_length                 11
inq_last_6mths              9
grade                       7
pub_rec                     5
home_ownership              5
verification_status         3
loan_status                 3
pub_rec_bankruptcies        3
term                        2
dtype: int64

In [302]:
lc_dkt

,LoanStatNew,Description
0,acc_now_delinq,The number of accounts on which the borrower i...
1,acc_open_past_24mths,Number of trades opened in past 24 months.
2,addr_state,The state provided by the borrower in the loan...
3,all_util,Balance to credit limit on all trades
4,annual_inc,The self-reported annual income provided by th...
...,...,...
148,settlement_amount,The loan amount that the borrower has agreed t...
149,settlement_percentage,The settlement amount as a percentage of the p...
150,settlement_term,The number of months that the borrower will be...
151,NaN,NaN


In [303]:
(df.isna().sum()/len(df)*100).sort_values()

loan_amnt                0.000000
funded_amnt              0.000000
funded_amnt_inv          0.000000
term                     0.000000
int_rate                 0.000000
installment              0.000000
grade                    0.000000
sub_grade                0.000000
annual_inc               0.000000
home_ownership           0.000000
issue_d                  0.000000
verification_status      0.000000
loan_status              0.000000
dti                      0.000000
addr_state               0.000000
purpose                  0.000000
out_prncp_inv            0.000000
revol_bal                0.000000
total_acc                0.000000
delinq_2yrs              0.000000
inq_last_6mths           0.000000
earliest_cr_line         0.000000
open_acc                 0.000000
pub_rec                  0.000000
out_prncp                0.000000
last_credit_pull_d       0.005036
title                    0.027696
revol_util               0.125891
pub_rec_bankruptcies     1.754916
emp_length    

In [304]:
#df.isna().sum()
col_dict={}
for col in df.columns:
    col_dict[col]=df[col].isna().sum()
    

In [305]:
for key, value in col_dict.items():
    if value < 39717:
        print(key)

loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
desc
purpose
title
addr_state
dti
delinq_2yrs
earliest_cr_line
inq_last_6mths
open_acc
pub_rec
revol_bal
revol_util
total_acc
out_prncp
out_prncp_inv
last_credit_pull_d
pub_rec_bankruptcies
